# MRI Processing Pipeline - Original Workflow Without iBEATv2
## Exact Published Pipeline Minus Steps 3-4

This notebook follows the **exact original pipeline** from `reFS_under25mo.sh` but removes only the iBEATv2-dependent steps.

**Input File:** `/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz`

### Original Pipeline Steps (from `reFS_under25mo.sh`):

1. ✅ **Standard FreeSurfer Initial Processing** - `recon-all -all -subjid ${sub}`
2. ✅ **Clean Up Files** - Remove certain files from initial FS run
3. ✅ **Prepare Infant FreeSurfer** - Convert orig.mgz to mprage.nii.gz
4. ✅ **Run Infant FreeSurfer** - `infant_recon_all --s ${sub} --age ${age}`
5. ❌ **~~iBEATv2 Merging~~** - ~~SKIP: `ibeat2aseg.m` and `aseg2wm.m`~~
6. ✅ **Resume FreeSurfer autorecon2** - `fs_autorecon2_end.sh` (use iFS segmentation)
7. ✅ **Finish with autorecon3** - `fs_autorecon3_wrap.sh`

### What Changes:

**Removed (Steps 5):**
```matlab
# SKIP THIS - No iBEATv2 available
matlab -nodesktop -nosplash -r "addpath(${m_fun}); ibeat2aseg(${m_ib}, ${m_ifp}, ${m_fp}); aseg2wm(${m_aseg}); exit;"
```

**Instead, Use:**
```bash
# Use infant FreeSurfer's native segmentation directly
cp ${ifp}/mri/aseg.mgz ${fp}/mri/aseg.presurf.mgz
cp ${ifp}/mri/wm.mgz ${fp}/mri/wm.mgz
```

**Everything else stays exactly the same as the published pipeline!**

In [ ]:
# Import required libraries
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.mplot3d import Axes3D
import subprocess
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Define paths - Following original script structure
INPUT_T1W = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz"
SUBJECT_ID = "sub-01_ses-03"
AGE_MONTHS = 6  # age for infant FS (required for <25 months)

# Original variable names from reFS_under25mo.sh
SUBJECTS_DIR = "/data02/share/bin-wu/data/human/brain/harvard_mri/processed/sandbox/freesurfer_output"
fp = os.path.join(SUBJECTS_DIR, SUBJECT_ID)  # FreeSurfer subject path

if_dir = "/data02/share/bin-wu/data/human/brain/harvard_mri/processed/sandbox/iFS"  # Infant FS directory
ifp = os.path.join(if_dir, SUBJECT_ID)  # Infant FS subject path

os.makedirs(SUBJECTS_DIR, exist_ok=True)
os.makedirs(if_dir, exist_ok=True)

print(f"FreeSurfer subject directory: {fp}")
print(f"Infant FreeSurfer directory: {ifp}")
print(f"Input file: {INPUT_T1W}")
print(f"Age: {AGE_MONTHS} months")

## Helper Functions

In [ ]:
def load_nifti(filepath):
    """Load NIfTI file and return image data and header."""
    img = nib.load(filepath)
    data = img.get_fdata()
    return data, img.affine, img.header

def plot_3d_slices(data, title="MRI Slices", figsize=(15, 5), cmap='gray', percentiles=(1, 99)):
    """Plot axial, sagittal, and coronal slices of 3D MRI data."""
    mid_x = data.shape[0] // 2
    mid_y = data.shape[1] // 2
    mid_z = data.shape[2] // 2
    vmin, vmax = np.percentile(data[data > 0], percentiles)
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    axes[0].imshow(np.rot90(data[mid_x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title(f'Sagittal (X={mid_x})')
    axes[0].axis('off')
    
    axes[1].imshow(np.rot90(data[:, mid_y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title(f'Coronal (Y={mid_y})')
    axes[1].axis('off')
    
    axes[2].imshow(np.rot90(data[:, :, mid_z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title(f'Axial (Z={mid_z})')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Load input image
if os.path.exists(INPUT_T1W):
    print("Loading original T1w image...")
    t1w_data, t1w_affine, t1w_header = load_nifti(INPUT_T1W)
    print(f"Image dimensions: {t1w_data.shape}")
    print(f"Voxel size: {t1w_header.get_zooms()[:3]} mm")
    plot_3d_slices(t1w_data, title="Original T1w Image")
    plt.show()

## STEP 1: Run Standard FreeSurfer Initial Processing

**Original command from `reFS_under25mo.sh:55`:**
```bash
recon-all -all -subjid ${sub} -nonuintensitycor
```

**Adapted for FreeSurfer 8.x (remove -nonuintensitycor):**

In [ ]:
print("""STEP 1: Standard FreeSurfer Initial Processing
="*70)

step1_cmd = f"""
export SUBJECTS_DIR={SUBJECTS_DIR}

# Original: recon-all -all -subjid {SUBJECT_ID} -nonuintensitycor
# Adapted for FS 8.x (remove -nonuintensitycor due to compatibility)

recon-all \
  -i {INPUT_T1W} \
  -subjid {SUBJECT_ID} \
  -all
"""

print("Command to run:")
print(step1_cmd)
print("\nExpected output files:")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/orig.mgz")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/nu.mgz")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/T1.mgz")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/brainmask.mgz")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/aseg.mgz")
print("  - $SUBJECTS_DIR/sub-01_ses-03/mri/transforms/")
print("\nProcessing time: 6-12 hours")

## STEP 2: Clean Up Files from Initial FreeSurfer Run

**Original command from `reFS_under25mo.sh:61`:**
```bash
rm ${fp}/mri/transforms/*
rm ${fp}/mri/orig_nu.mgz
rm ${fp}/mri/mri_nu_correct.mni.log
```

**Adapted for FreeSurfer 8.x:**
- `orig_nu.mgz` and `mri_nu_correct.mni.log` don't exist in FS 8.x
- Only need to clear `transforms/` directory

In [ ]:
print("""STEP 2: Remove Files Based on Initial FS Run
="*70)

step2_cmd = f"""
# Backup transforms directory (safer than rm)
mv {fp}/mri/transforms {fp}/mri/transforms.bak
mkdir -p {fp}/mri/transforms

# Note: orig_nu.mgz and mri_nu_correct.mni.log don't exist in FS 8.x
# Original script removed them but they're not present in FS 8.x
"""

print("Command to run:")
print(step2_cmd)
print("\nWhy this step is needed:")
print("  - Adult transforms must be removed")
print("  - Infant FreeSurfer will create age-appropriate transforms")
print("  - Prevents mixing adult and infant atlas registrations")

## STEP 3: Prepare for Infant FreeSurfer

**Original commands from `reFS_under25mo.sh:65-67`:**
```bash
mkdir -p ${ifp}
mri_convert -i ${fp}/mri/orig.mgz -o ${ifp}/mprage.nii.gz
```

In [ ]:
print("""STEP 3: Setting Up for Infant FreeSurfer
="*70)

step3_cmd = f"""
# Create infant FreeSurfer directory
mkdir -p {ifp}

# Convert FreeSurfer orig.mgz to infant FreeSurfer format
mri_convert -i {fp}/mri/orig.mgz -o {ifp}/mprage.nii.gz
"""

print("Command to run:")
print(step3_cmd)
print("\nWhat this does:")
print("  - Creates directory structure for infant FreeSurfer")
print("  - Converts MGZ format to NIfTI format (mprage.nii.gz)")
print("  - Preserves original FreeSurfer output")

## STEP 4: Run Infant FreeSurfer

**Original command from `reFS_under25mo.sh:72` via `iFS_wrap.sh:14`:**
```bash
infant_recon_all --s ${sub} --age ${age}
```

Uses age-specific infant atlases for better subcortical segmentation.

In [ ]:
print(f"""STEP 4: Running Infant FreeSurfer (Age: {AGE_MONTHS} months)
="*70)

step4_cmd = f"""
# Set infant FreeSurfer environment
# Note: Adjust FREESURFER_HOME path to your infant FreeSurfer installation
export FREESURFER_HOME=/path/to/infant_freesurfer
source $FREESURFER_HOME/SetUpFreeSurfer.sh
export SUBJECTS_DIR={if_dir}

# Run infant FreeSurfer with age parameter
infant_recon_all --s {SUBJECT_ID} --age {AGE_MONTHS}
"""

print("Command to run:")
print(step4_cmd)
print("\nExpected output files:")
print(f"  - {ifp}/mri/aseg.mgz (infant-specific segmentation)")
print(f"  - {ifp}/mri/brainmask.mgz (infant-specific brain mask)")
print(f"  - {ifp}/mri/wm.mgz (white matter mask)")
print(f"  - {ifp}/mri/transforms/talairach*.xfm (infant atlas transforms)")
print("\nProcessing time: 4-8 hours")
print("\nKey advantages of infant FreeSurfer:")
print("  - Age-appropriate atlases (0-24 months)")
print("  - Better subcortical structure segmentation")
print("  - Handles poor GM/WM contrast in infant brains")

## STEP 5: ~~iBEATv2 Merging~~ → Use Infant FreeSurfer Segmentation Directly

**Original Step (SKIPPED):**
```bash
# Original from reFS_under25mo.sh:80
matlab -nodesktop -nosplash -r "addpath(${m_fun}); ibeat2aseg(${m_ib}, ${m_ifp}, ${m_fp}); aseg2wm(${m_aseg}); exit;"
```

**Replacement (No iBEATv2):**
```bash
# Copy infant FreeSurfer segmentation directly
cp ${ifp}/mri/aseg.mgz ${fp}/mri/aseg.presurf.mgz
cp ${ifp}/mri/wm.mgz ${fp}/mri/wm.mgz
```

### What Changes:

| Original Pipeline | This Pipeline (No iBEATv2) |
|-------------------|----------------------------|
| Uses merged iBEATv2+iFS segmentation | Uses infant FreeSurfer native segmentation |
| MATLAB scripts merge tissue labels | Direct file copy |
| Cortical: iBEATv2 labels | Cortical: Infant FreeSurfer labels |
| Subcortical: iFS labels | Subcortical: iFS labels (same!) |

**Impact:**
- ✅ Subcortical structures: No change (both use iFS)
- ⚠️ Cortical GM/WM boundary: Slightly less accurate (no iBEATv2 refinement)
- ✅ Surface reconstruction: Uses same FreeSurfer commands

In [ ]:
print("""STEP 5: Copy Infant FreeSurfer Segmentation (Replaces iBEATv2 Merging)
="*70)

step5_cmd = f"""
# ORIGINAL PIPELINE (requires iBEATv2 + MATLAB):
# matlab -nodesktop -nosplash -r "...; ibeat2aseg(...); aseg2wm(...); exit;"

# THIS PIPELINE (no iBEATv2 required):
# Use infant FreeSurfer segmentation directly

cp {ifp}/mri/aseg.mgz {fp}/mri/aseg.presurf.mgz
cp {ifp}/mri/wm.mgz {fp}/mri/wm.mgz
"""

print("Command to run:")
print(step5_cmd)
print("\nWhat we're doing differently:")
print("  ✗ Skip: MATLAB ibeat2aseg.m (merges iBEATv2 + iFS)")
print("  ✗ Skip: MATLAB aseg2wm.m (generates WM from merged seg)")
print("  ✓ Use: Infant FreeSurfer's native segmentation")
print("  ✓ Use: Infant FreeSurfer's native WM mask")
print("\nOutput files created:")
print(f"  - {fp}/mri/aseg.presurf.mgz (from iFS aseg.mgz)")
print(f"  - {fp}/mri/wm.mgz (from iFS wm.mgz)")
print("\nThese files have the SAME NAMES as the original pipeline")
print("So all subsequent steps work identically!")

## STEP 6: Resume FreeSurfer Processing (autorecon2-end)

**Original commands from `reFS_under25mo.sh:85-86` via `fs_autorecon2_end.sh`:**

These are the EXACT commands from the original pipeline. They work unchanged because we created `aseg.presurf.mgz` and `wm.mgz` with the same names.

In [ ]:
print("""STEP 6: Finish FreeSurfer autorecon2-wm Pipeline
="*70)

# These are the EXACT commands from fs_autorecon2_end.sh
step6_cmd = f"""
export SUBJECTS_DIR={SUBJECTS_DIR}

# First, copy infant FreeSurfer transforms (original line 85)
cp {ifp}/mri/transforms/talairach*xfm {fp}/mri/transforms

# Then run all commands from fs_autorecon2_end.sh
cd {fp}/mri

# Resume -autorecon1 using iFS files
mri_nu_correct.mni --i orig.mgz --o nu.mgz --uchar transforms/talairach.xfm --n 2 --ants-n4
mri_normalize -g 1 -seed 1234 -mprage nu.mgz T1.mgz
mri_mask {fp}/mri/T1.mgz {ifp}/mri/brainmask.mgz {fp}/mri/brainmask.mgz
mri_em_register -uns 3 -mask brainmask.mgz nu.mgz $FREESURFER_HOME/average/RB_all_2020-01-02.gca transforms/talairach.lta
mri_ca_normalize -c ctrl_pts.mgz -mask brainmask.mgz nu.mgz $FREESURFER_HOME/average/RB_all_2020-01-02.gca transforms/talairach.lta norm.mgz
mri_normalize -seed 1234 -mprage -aseg aseg.presurf.mgz -mask brainmask.mgz norm.mgz brain.mgz

# Perform most functions of -autorecon2
mri_mask -T 5 brain.mgz brainmask.mgz brain.finalsurfs.mgz
mri_fill -a ../scripts/ponscc.cut.log -xform transforms/talairach.lta -segmentation aseg.presurf.mgz wm.mgz filled.mgz -ctab $FREESURFER_HOME/SubCorticalMassLUT.txt
cp filled.mgz filled.auto.mgz
mri_pretess filled.mgz 255 norm.mgz filled-pretess255.mgz
mri_tessellate filled-pretess255.mgz 255 ../surf/lh.orig.nofix
rm -f filled-pretess255.mgz
mri_pretess filled.mgz 127 norm.mgz filled-pretess127.mgz
mri_tessellate filled-pretess127.mgz 127 ../surf/rh.orig.nofix
rm -f filled-pretess127.mgz

cd {fp}/surf

mris_extract_main_component lh.orig.nofix lh.orig.nofix
mris_extract_main_component rh.orig.nofix rh.orig.nofix
mris_smooth -nw -seed 1234 lh.orig.nofix lh.smoothwm.nofix
mris_smooth -nw -seed 1234 rh.orig.nofix rh.smoothwm.nofix
mris_inflate -no-save-sulc lh.smoothwm.nofix lh.inflated.nofix
mris_inflate -no-save-sulc rh.smoothwm.nofix rh.inflated.nofix
mris_sphere -q -p 6 -a 128 -seed 1234 -in 3000 lh.inflated.nofix lh.qsphere.nofix
mris_sphere -q -p 6 -a 128 -seed 1234 -in 3000 rh.inflated.nofix rh.qsphere.nofix

cp lh.orig.nofix lh.orig
cp rh.orig.nofix rh.orig
cp lh.qsphere.nofix lh.qsphere
cp rh.qsphere.nofix rh.qsphere

mris_topo_fixer -mgz -warnings {SUBJECT_ID} lh
mris_topo_fixer -mgz -warnings {SUBJECT_ID} rh

mv lh.orig_corrected lh.orig.premesh
mv rh.orig_corrected rh.orig.premesh
rm lh.orig rh.orig lh.orig_corrected rh.orig_corrected

mris_euler_number lh.orig.premesh
mris_euler_number rh.orig.premesh
mris_remesh --remesh --iters 3 --input lh.orig.premesh --output lh.orig
mris_remesh --remesh --iters 3 --input rh.orig.premesh --output rh.orig
mris_remove_intersection lh.orig lh.orig
mris_remove_intersection rh.orig rh.orig
mris_autodet_gwstats --o autodet.gw.stats.lh.dat --i ../mri/brain.finalsurfs.mgz --wm ../mri/wm.mgz --surf lh.orig.premesh
mris_autodet_gwstats --o autodet.gw.stats.rh.dat --i ../mri/brain.finalsurfs.mgz --wm ../mri/wm.mgz --surf rh.orig.premesh

mris_make_surfaces -output .preaparc -soap -orig_white orig -aseg aseg.presurf -cover_seg {fp}/mri/aseg.presurf.mgz -noaparc -whiteonly -mgz -T1 brain.finalsurfs {SUBJECT_ID} lh
mris_make_surfaces -output .preaparc -soap -orig_white orig -aseg aseg.presurf -cover_seg {fp}/mri/aseg.presurf.mgz -noaparc -whiteonly -mgz -T1 brain.finalsurfs {SUBJECT_ID} rh

mri_label2label --label-cortex lh.white.preaparc ../mri/aseg.presurf.mgz 0 ../label/lh.cortex.label
mri_label2label --label-cortex lh.white.preaparc ../mri/aseg.presurf.mgz 1 ../label/lh.cortex+hipamyg.label
mri_label2label --label-cortex rh.white.preaparc ../mri/aseg.presurf.mgz 0 ../label/rh.cortex.label
mri_label2label --label-cortex rh.white.preaparc ../mri/aseg.presurf.mgz 1 ../label/rh.cortex+hipamyg.label
mris_smooth -n 3 -nw -seed 1234 lh.white.preaparc lh.smoothwm
mris_smooth -n 3 -nw -seed 1234 rh.white.preaparc rh.smoothwm
mris_inflate lh.smoothwm lh.inflated
mris_inflate rh.smoothwm rh.inflated
mris_curvature -w -seed 1234 lh.white.preaparc
ln -s lh.white.preaparc.H lh.white.H
ln -s lh.white.preaparc.K lh.white.K
mris_curvature -seed 1234 -thresh .999 -n -a 5 -w -distances 10 10 lh.inflated
mris_curvature -w -seed 1234 rh.white.preaparc
ln -s rh.white.preaparc.H rh.white.H
ln -s rh.white.preaparc.K rh.white.K
mris_curvature -seed 1234 -thresh .999 -n -a 5 -w -distances 10 10 rh.inflated
"""

print("Commands to run (EXACT copy from fs_autorecon2_end.sh):")
print(step6_cmd)
print("\n" + "="*70)
print("Processing time: 8-15 hours")
print("These are IDENTICAL to the original pipeline!")
print("="*70)

## STEP 7: Finish with autorecon3

**Original commands from `reFS_under25mo.sh:91` via `fs_autorecon3_wrap.sh`:**

Again, these are the EXACT commands from the original pipeline.

In [ ]:
print("""STEP 7: Run FreeSurfer autorecon3 with Adjustments
="*70)

# Get path to expert.opts
fun = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'peer-review/1.Structure')

step7_cmd = f"""
export SUBJECTS_DIR={SUBJECTS_DIR}

# Commands from fs_autorecon3_wrap.sh (exact copy)
# Use expert options to control pial surface placement
recon-all -autorecon3 -subjid {SUBJECT_ID} -expert {fun}/expert.opts

# Use infant-specific parameters for pial surface (from iFS)
cd {fp}/surf
mris_make_surfaces -grad_dir 1 -intensity .3 -output .tmp -pial_offset .25 -nowhite -noaparc -aseg aseg.presurf -cover_seg {fp}/mri/aseg.presurf.mgz -orig_pial white {SUBJECT_ID} lh
mris_make_surfaces -grad_dir 1 -intensity .3 -output .tmp -pial_offset .25 -nowhite -noaparc -aseg aseg.presurf -cover_seg {fp}/mri/aseg.presurf.mgz -orig_pial white {SUBJECT_ID} rh

mv {fp}/surf/lh.pial.tmp {fp}/surf/lh.pial.T1
mv {fp}/surf/rh.pial.tmp {fp}/surf/rh.pial.T1
ln -sf {fp}/surf/lh.pial.T1 {fp}/surf/lh.pial
ln -sf {fp}/surf/rh.pial.T1 {fp}/surf/rh.pial

# Run rest of autorecon3 from T2pial step
recon-all -autorecon3-T2pial -noT2pial -subjid {SUBJECT_ID}
"""

print("Commands to run (EXACT copy from fs_autorecon3_wrap.sh):")
print(step7_cmd)
print("\n" + "="*70)
print("Processing time: 3-6 hours")
print("\nInfant-specific parameters used:")
print("  -grad_dir 1: Gradient direction for infant brains")
print("  -intensity .3: Lower threshold for infant GM/WM contrast")
print("  -pial_offset .25: Smaller offset for thinner infant cortex")
print("\nThese are IDENTICAL to the original pipeline!")
print("="*70)

## Visualization Functions

(Same as original notebook)

In [ ]:
def plot_segmentation(seg_data, title="Segmentation", figsize=(15, 5)):
    """Plot segmentation with color map."""
    n_labels = int(seg_data.max()) + 1
    cmap = plt.cm.get_cmap('tab20', n_labels)
    
    mid_x = seg_data.shape[0] // 2
    mid_y = seg_data.shape[1] // 2
    mid_z = seg_data.shape[2] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    axes[0].imshow(np.rot90(seg_data[mid_x, :, :]), cmap=cmap, interpolation='nearest')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(np.rot90(seg_data[:, mid_y, :]), cmap=cmap, interpolation='nearest')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(np.rot90(seg_data[:, :, mid_z]), cmap=cmap, interpolation='nearest')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Visualize final segmentation
aseg_path = os.path.join(fp, "mri", "aseg.presurf.mgz")
if os.path.exists(aseg_path):
    print("Loading final segmentation...")
    aseg_data, _, _ = load_nifti(aseg_path)
    plot_segmentation(aseg_data, title="Final Segmentation (Infant FreeSurfer)")
    plt.show()
else:
    print(f"Segmentation not found at: {aseg_path}")
    print("Run steps 1-5 first.")

## Summary: What Changed from Original Pipeline

### ONLY ONE DIFFERENCE:

**Step 5 - Segmentation Preparation:**

| Original Pipeline | This Pipeline |
|-------------------|---------------|
| ```matlab -r "ibeat2aseg(...); aseg2wm(...);"``` | ```cp iFS/aseg.mgz FS/aseg.presurf.mgz```<br>```cp iFS/wm.mgz FS/wm.mgz``` |
| Merges iBEATv2 + iFS | Uses iFS directly |
| Requires MATLAB | No MATLAB needed |
| Requires iBEATv2 preprocessing | No iBEATv2 needed |

### EVERYTHING ELSE IS IDENTICAL:

✅ Step 1: Same FreeSurfer command (adapted for FS 8.x)
✅ Step 2: Same file cleanup (adapted for FS 8.x)
✅ Step 3: Same conversion to infant FS format
✅ Step 4: Same infant FreeSurfer command
✅ Step 6: **EXACTLY THE SAME** fs_autorecon2_end.sh commands
✅ Step 7: **EXACTLY THE SAME** fs_autorecon3_wrap.sh commands

### Impact on Results:

- **Subcortical structures**: No change (both use infant FreeSurfer)
- **Cortical surfaces**: Minimal change (surface reconstruction identical)
- **Cortical GM/WM boundary**: Slightly less refined (no iBEATv2)
- **Overall accuracy**: Excellent for most analyses

### When This Pipeline is Appropriate:

✅ You don't have iBEATv2
✅ You don't have MATLAB
✅ Your focus is on subcortical structures
✅ Your focus is on surfaces and cortical parcellation
✅ Infant FreeSurfer accuracy is sufficient for your cortical needs

### When to Use Full Pipeline Instead:

❌ You need maximum cortical GM/WM boundary accuracy in 0-12 month olds
❌ You're specifically studying fine cortical development
❌ You have iBEATv2 and MATLAB readily available
❌ You're replicating a published study using the full pipeline